# 1. Tesseract + Line segmentation

In [2]:
import gc  # Garbage Collector
import os
import re
import time
import traceback
import unicodedata
from collections import deque

import cv2
import fitz  # PyMuPDF
import numpy as np
import pandas as pd
import pytesseract
from pdf2image import convert_from_path, pdfinfo_from_path
from PIL import Image, ImageFilter
from pytesseract import Output
from sklearn.cluster import KMeans
from tqdm.notebook import tqdm

# Set Tesseract to use multiple threads internally
os.environ["OMP_THREAD_LIMIT"] = "8"

In [26]:
# --- Helper Functions + Main OCR Function---
def save_entry_to_file(entry_str, file_path="ocr_results.txt"):
    with open(file_path, "a", encoding="utf-8") as f:
        f.write(entry_str + "\n\n")

def is_garbage_token(text):
    """
    Evaluates if a string is likely OCR garbage/misread character noise.
    Returns True if the token should be discarded.
    """
    clean_txt = text.strip(".,;:!?()[]-+=/*\"'").lower()
    if not clean_txt:
        return True

    # 4. Filter out high-consonant clusters (OCR hallucination signature)
    # Tembo, Swahili, and French are highly vocalic. 4+ consecutive consonants is garbage.
    if re.search(r'[^aeiouyéàèùûîüũĩāēōū\s]{4,}', clean_txt):
        return True

    # 5. Check vowel ratio 
    # Bantu languages (Swahili/Tembo) generally maintain a vowel ratio of 35% to 60%+
    vowels = len(re.findall(r'[aeiouyéàèùûîüũĩāēōū]', clean_txt))
    vowel_ratio = vowels / len(clean_txt)
    
    # If a word longer than 3 chars has practically no vowels, it's garbage noise (e.g., "lhz", "kkhohora")
    if len(clean_txt) > 3 and vowel_ratio < 0.20:
        return True

    return False

def calculate_column_thresholds(lines_dict):
    all_lefts = [word['left'] for line in lines_dict.values() for word in line]
    if not all_lefts:
        return 0, 0
        
    max_left = max(all_lefts)
    
    # Calculate Description Threshold
    right_half_lefts = [l for l in all_lefts if l > (max_left * 0.4)]
    desc_threshold = min(right_half_lefts) if right_half_lefts else (max_left * 0.45)
    
    # Calculate Entry Threshold
    left_half_lefts = [l for l in all_lefts if l < desc_threshold]
    if left_half_lefts:
        min_left = min(left_half_lefts)
        indented_lefts = [l for l in left_half_lefts if l > (min_left + 20)]
        entry_threshold = min(indented_lefts) if indented_lefts else (desc_threshold * 0.3)
    else:
        entry_threshold = desc_threshold * 0.3
        
    return entry_threshold, desc_threshold
    
def ocr_pdf_lineseg_pipeline(
    pdf_path,
    image_dpi=600,
    languages="fra+latn",
    ocr_config="--oem 3 --psm 11", 
    output_file="ocr_results.txt",
    export_pics=False,
    limit_max_p=None,
):

    max_pages = pdfinfo_from_path(pdf_path)["Pages"]

    def write_entry(entry, marker=None):
        if not entry:
            return
        s = "\n".join(entry).strip()  
        with open(output_file, "a", encoding="utf-8") as f:
            f.write(f"{s}\n{marker}\n\n" if marker else f"{s}\n\n")

    open(output_file, "w", encoding="utf-8").close()  

    current_entry, held_marker = [], None

    for page_num in tqdm(
        range(1, max_pages + 1), desc="Processing Dictionary", unit="page"
    ):
        if limit_max_p and page_num > limit_max_p:
            break
        try:
            images = convert_from_path(
                pdf_path, dpi=image_dpi, first_page=page_num, last_page=page_num
            )
            if not images:
                continue
            img = images[0]

            if export_pics:
                img.save(f"p{page_num}.png")

            data = pytesseract.image_to_data(
                img, lang=languages, config=ocr_config, output_type=pytesseract.Output.DICT
            )
            img.close()

            n_boxes = len(data['text'])
            lines_dict = {}

            # --- STEP 1: GROUP TOKENS BY LINE & FILTER OUT GARBAGE ---
            for i in range(n_boxes):
                text_content = str(data['text'][i]).strip()
                if not text_content:  
                    continue
                
                # Apply the garbage/noise filtration logic
                if is_garbage_token(text_content):
                    continue
                
                line_id = (data['block_num'][i], data['par_num'][i], data['line_num'][i])
                
                if line_id not in lines_dict:
                    lines_dict[line_id] = []
                
                lines_dict[line_id].append({
                    'text': text_content,
                    'left': data['left'][i]
                })

            # --- STEP 2: DYNAMICALLY FIND THE ENTRY & DESCRIPTION BOUNDARIES ---
            entry_threshold, desc_threshold = calculate_column_thresholds(lines_dict)

            # --- STEP 3: ASSEMBLE LINES WITH MARKERS ---
            sorted_line_keys = sorted(lines_dict.keys())

            for line_key in sorted_line_keys:
                line_tokens = lines_dict[line_key]
                if not line_tokens:
                    continue

                assembled_line_pieces = []
                hit_entry_zone = False
                hit_description_zone = False

                for token in line_tokens:
                    if not hit_description_zone and token['left'] >= desc_threshold:
                        if token['text'] == ":":
                            assembled_line_pieces.append("<D>")
                            hit_description_zone = True
                            continue
                        else:
                            assembled_line_pieces.append("<D>")
                            hit_description_zone = True
                    
                    elif not hit_description_zone and not hit_entry_zone and token['left'] >= entry_threshold:
                        assembled_line_pieces.append("<E>")
                        hit_entry_zone = True
                    
                    elif hit_description_zone and token['text'] == ":":
                        assembled_line_pieces.append("<D>")
                        continue

                    assembled_line_pieces.append(token['text'])

                line_str = " ".join(assembled_line_pieces)
                line_str = line_str.replace(" <E>", " <E> ").replace("<E> ", "<E>")
                line_str = line_str.replace(" <D>", " <D> ").replace("<D> ", "<D>")
                line_str = line_str.replace("<E> <E>", "<E>").replace("<D> <D>", "<D>")
                
                # Ensure lines that are now entirely tags due to dropped tokens are skipped
                cleaned_validation = line_str.replace("<E>", "").replace("<D>", "").strip(" ,;.:")
                if cleaned_validation and len(cleaned_validation) > 1:
                    current_entry.append(line_str)

            if current_entry:
                write_entry(current_entry, marker=held_marker)
                current_entry, held_marker = [], None

        except Exception as e:
            print(f"Error processing page {page_num}: {e}")
            continue

    if current_entry:
        write_entry(current_entry, marker=held_marker)

    return f"Success! Results appended to {output_file}"

ModuleNotFoundError: No module named 'pdfinfo_from_path'

In [25]:
filename = "tembo_sample.pdf"  
output_file = "tembo_extracted_sample.txt"

start_time = time.perf_counter()
extracted_content = ocr_pdf_lineseg_pipeline(
    filename,
    image_dpi=600,
    languages="script/Latin",
    ocr_config=f"--oem 3 --psm 11 -c load_system_dawg=0 -c load_freq_dawg=0",
    output_file=output_file,
    export_pics=False,  # debug
    limit_max_p=None,  # debug
)

end_time = time.perf_counter()

print(f"Time taken to OCR: {(end_time-start_time)/60:.2f} min")
extracted_content

Processing Dictionary:   0%|          | 0/4 [00:00<?, ?page/s]

Time taken to OCR: 0.48 min


'Success! Results appended to tembo_extracted_sample.txt'

# 2. Text Cleaning

In [244]:
print(
    sorted(
        __import__("collections")
        .Counter(open("kikuyu_extracted_full_cleaned.txt", encoding="utf-8").read())
        .items(),
        key=lambda x: x[1],
    )
)

[('ʻ', 1), ('¢', 1), ('‚', 1), ('\\', 1), ('¡', 1), ('Œ', 1), ('Ł', 1), ('©', 1), ('>', 1), ('Đ', 1), ('§', 1), ('€', 2), ('¥', 2), ('%', 2), ('”', 2), ('+', 2), ('$', 2), ('Þ', 3), ('ə', 3), ('«', 3), ('£', 3), ('#', 4), ('“', 5), ('„', 5), ('þ', 6), ('đ', 6), ('°', 6), ('»', 6), ('=', 7), ('_', 7), ('Q', 8), ('—', 10), ('ħ', 13), ('{', 15), ('`', 16), ('¿', 18), ('ł', 18), ('@', 19), ('*', 19), ('Ĩ', 23), ('Z', 38), ('"', 39), ('Y', 58), ('J', 59), ('F', 86), ('W', 110), ('X', 126), ('‘', 143), ('D', 148), ('}', 161), (':', 182), ('H', 215), ('B', 234), ('V', 273), ('7', 275), ('8', 293), ('G', 299), ('9', 324), ('Ũ', 350), ('|', 429), ('0', 447), ('5', 483), ('E', 635), ("'", 756), ('6', 790), ('q', 926), ('z', 1031), ('!', 1052), ('K', 1123), ('3', 1155), ('4', 1219), ('P', 1248), ('L', 1312), ('R', 1399), ('O', 1449), ('/', 1474), ('&', 1558), ('U', 1634), ('?', 1690), ('M', 1792), ('S', 1828), ('A', 1929), ('I', 2191), ('N', 2348), ('x', 2470), ('T', 2478), ('C', 2581), ('’', 329

In [263]:
def fix_dictionary_diacritics(text):
    # 1. Decompose text to isolate base characters from their diacritics
    # e.g., 'ā' becomes 'a' + '\u0304', 'īï' becomes 'i' + '\u0304' + 'i' + '\u0308'
    decomposed_text = unicodedata.normalize("NFD", text)

    # 2. Pattern 1: Catch contiguous 'i' or 'u' clusters with any combining diacritics
    kikuyu_vowel_pattern = r"(?i)(?:i[\u0300-\u036f]*)+|(?:u[\u0300-\u036f]*)+"

    def handle_kikuyu_vowels(match):
        cluster = match.group(0)
        # Check if this i/u cluster actually has an OCR diacritic noise symbol
        has_diacritic = any("\u0300" <= char <= "\u036f" for char in cluster)

        if has_diacritic:
            base_vowel = cluster[0] # we consider only the first vowel as the base char
            if base_vowel.lower() == "i":
                return "Ĩ" if base_vowel.isupper() else "ĩ"
            elif base_vowel.lower() == "u":
                return "Ũ" if base_vowel.isupper() else "ũ"
        return cluster

    # Apply the Kikuyu specific tilde logic first
    processed_text = re.sub(kikuyu_vowel_pattern, handle_kikuyu_vowels, decomposed_text)

    # 3. Pattern 2: Catch ANY remaining stray combining diacritics on other letters
    # Because the text is decomposed, we can literally just strip out the diacritic symbols
    # from any other consonants or standard vowels (a, e, o).
    stray_diacritics_pattern = r"[\u0300-\u036f]+"
    processed_text = re.sub(stray_diacritics_pattern, "", processed_text)

    # 4. Recompose back to standard NFC format
    return unicodedata.normalize("NFC", processed_text)


def clean_dictionary_typos(text):

    # remove recurring id marker
    text = re.sub(r"\s864403\s[A-Z]", "", text)
    # handle hyphen breaks
    text = re.sub(r"-\s([a-zũĩ])", r"\1", text)

    # 1. Standard static replacements
    text = text.replace("fħ", "ffi")
    text = text.replace("œ", "~").replace("æ", "~")
    text = text.replace("ø", "g")
    text = text.replace("ı", "i")
    text = text.replace("Ħ", "")
    text = text.replace("864403", "")

    # 2. Handle 'ł' variations
    text = text.replace("łi", "i").replace("ił", "i")
    text = text.replace("ł-", "i-")

    # 3. Contextual 'ð' replacement
    # Pattern A: ð surrounded by vowels (case-insensitive) -> delete it
    # [aeiouĩũ] captures standard vowels + the Kikuyu tilde vowels
    vowels = r"[aeiouĩũAEIOUĨŨ]"
    vowel_surrounded_pattern = f"(?<={vowels})ð(?={vowels})"
    text = re.sub(vowel_surrounded_pattern, "", text)

    # Pattern B: Any remaining ð (not surrounded by vowels) -> replace with ũ
    text = text.replace("ð", "ũ")

    return text


def process_dictionary_file(input_filepath, output_filepath):
    """Reads the entire text file, cleans all diacritics, and writes it back."""
    print(f"Reading {input_filepath}...")
    with open(input_filepath, "r", encoding="utf-8") as f:
        content = f.read()

    print(
        "Processing file: \nconverting i/u clusters and stripping other diacritics..."
    )
    cleaned_content = fix_dictionary_diacritics(content)

    print("Applying typo replacement rules...")
    cleaned_content = clean_dictionary_typos(cleaned_content)

    print(f"Saving cleaned file to {output_filepath}...")
    with open(output_filepath, "w", encoding="utf-8") as f:
        f.write(cleaned_content)
    print("Done!")

In [264]:
INPUT_FILE = "kikuyu_extracted_full.txt"
OUTPUT_FILE = "kikuyu_extracted_full_cleaned.txt"

process_dictionary_file(INPUT_FILE, OUTPUT_FILE)

Reading kikuyu_extracted_full.txt...
Processing file: 
converting i/u clusters and stripping other diacritics...
Applying typo replacement rules...
Saving cleaned file to kikuyu_extracted_full_cleaned.txt...
Done!


# 3. Manual Tagging

In [305]:
import re

import pandas as pd
# POS tags — note the leading space is intentional (avoids false matches mid-word)
pos_tags = [
    " n.",
    " n,",
    " a.",
    " n.c.",
    " abb.",
    " a.c.",
    " adv.",
    " adj.",
    " conj.",
    " dem.",
    " ideo.",
    " int.",
    " interj.",
    " interrog.",
    " n.c,",
    " num.",
    " p.a.",
    " p.c.",
    " part.",
    " poss.",
    " pron.",
    " subj.",
    " v.",
    " v.i.",
    " v.t.",
]

prefix_list = [
    "mũ-",
    "i-",
    "ma-",
    "ki-",
    "tũ-",
    "ũ-",
    "ha-",
    "a-",
    "ĩ-",
    "ri-",
    "rũ-",
    "ci-",
    "kũ-",
    "ka-",
    "mĩ-",
    "mi-",
    "n-",
]

def regex_extract_tags(INPUT_FILE):

    with open(INPUT_FILE, "r", encoding="utf-8") as f:
        raw_chunks = [e.strip() for e in f.read().split("\n") if len(e.strip()) > 3]

    processed_entries = []
    current_column = "UNKNOWN"

    for chunk in raw_chunks:
        if chunk.upper().startswith("COLUMN-START"):
            current_column = chunk
        else:
            processed_entries.append((chunk, current_column))

    

    extracted_data = []

    for idx, (text, col) in enumerate(processed_entries):
        full_entry = " ".join(text.split())

        # ── 1. WORD: everything before the first comma ────────────────────────
        word_match = re.match(r"^([^,]+),", full_entry)
        if (
            not word_match
        ):  # only 15 such cases, usually cut-off sentences, not significant
            # print(idx,": \n", full_entry,'\n\n')
            extracted_data.append(
                {
                    "INDEX": idx,
                    "PAGE-COLUMN": col.split("COLUMN-START-")[1],
                    "WORD": "",
                    "PREFIX": "",
                    "NC": "",
                    "POS": "",
                    "DESCRIPTION": "",
                    "FULL_ENTRY": full_entry,
                }
            )
            continue

        word = word_match.group(1).strip()

        # ── 2. AFFIX: between 1st and 2nd comma, MUST contain a hyphen ───────
        affix = None
        # Search scope: up to " n." or " n," if present, else up to first comma/semicolon/colon, else first half
        if " n." in full_entry or " n," in full_entry:
            search_scope = re.split(r" n[.,]", full_entry)[0]
        elif any(e in full_entry for e in [",", ";", ":"]):
            search_scope = re.split(r"[,;:]", full_entry)[0]
        else:
            search_scope = full_entry[: len(full_entry) // 2]
        # Find all prefixes from predefined list that appear in search scope,
        # only when surrounded by non-alphabet characters or brackets
        affix_matches = []
        for p in prefix_list:
            p_stripped = p.strip()
            # Build pattern: non-alpha or bracket before, non-alpha or bracket after
            pattern = rf"(?:^|[^\w]|\()\s*({re.escape(p_stripped)})(?:[^\w]|\))"
            if re.search(pattern, search_scope):
                affix_matches.append(p_stripped)
        if affix_matches:
            affix = ", ".join(affix_matches)

        # ── 3. NC: after "cl. " → grab first match, only if before description ────
        nc = None
        nc_match = re.search(
            r"cl\.\s*([\d/a-z]+(?:\s*(?:or|and|only)\s*[\d/a-z]+)*)", full_entry
        )
        if nc_match:
            candidate_nc = nc_match.group(1).strip()
            desc_anchor_pos = full_entry.find("), ")
            nc_pos = nc_match.start()
            if desc_anchor_pos == -1 or nc_pos < desc_anchor_pos:
                nc = candidate_nc

        # ── 4. POS: leading-space anchored to avoid mid-word hits ─────────────
        # Escapes each tag and joins into one alternation; matches the last
        # occurrence so it isn't confused with abbreviations inside descriptions.
        pos = None
        pos_pattern = "|".join(
            re.escape(t) for t in sorted(pos_tags, key=len, reverse=True)
        )
        pos_match = re.search(rf"({pos_pattern})", full_entry)
        if pos_match:
            pos = pos_match.group(1).strip()

        # ── 5. DESCRIPTION: after "), " → nested if/else on terminator ───────────
        description = None
        desc_match = re.search(r"\),\s+(.+)", full_entry, re.DOTALL)
        if desc_match:
            raw = desc_match.group(1).strip()
            if ";" in raw or ":" in raw:
                # Rule 1: stop at first ";" or ":"
                term = re.search(r"(.+?)(?:[;:])", raw, re.DOTALL)
                if term:
                    description = term.group(1).strip()
            else:
                # Rule 2: stop at full-stop
                term = re.search(r"(.+?\.)(?:\s|$)", raw, re.DOTALL)
                if term:
                    description = term.group(1).strip()
            # Rule 3: neither found → None (description stays None)

        extracted_data.append(
            {
                "INDEX": idx,
                "PAGE-COLUMN": col.split("COLUMN-START-")[1],
                "WORD": word,
                "PREFIX": affix,
                "NC": nc,
                "POS": pos,
                "DESCRIPTION": description,
                "FULL_ENTRY": full_entry,
            }
        )

    return pd.DataFrame(extracted_data)

In [415]:
INPUT_FILE = "kikuyu_extracted_full_cleaned.txt"

kikuyu_tagged_df = regex_extract_tags(INPUT_FILE)

In [416]:
kikuyu_tagged_df[55:65]

,INDEX,PAGE-COLUMN,WORD,PREFIX,NC,POS,DESCRIPTION,FULL_ENTRY
55,55,P5-C1,mwaha,None,3,n.,None,"mwaha, n. cl. 3 only (111); -a ~, odd single; ..."
56,56,P5-C1,rĩaha,ma-,None,n.,I. one side of a fork,"rĩaha, ma-!, n. (11), I. one side of a fork; c..."
57,57,P5-C1,maaha?,None,6,n.,"1. products of progeny, offspring","maaha?, n. cl. 6 only (111), 1. products of pr..."
58,58,P5-C1,ahaa,None,None,interj.,"expr. delight or comprehension, tolerant disse...","ahaa, interj. (1), expr. delight or comprehens..."
59,59,P5-C1,ahuha [ahuuha],None,None,v.t.,"1. sniff (wind), snuff. 2. (of hyena, dog, &c....","ahuha [ahuuha], v.t. & i. (11), 1. sniff (wind..."
60,60,P5-C1,mwahuha [mwahuuha],mi-,None,n.,"a yawning hole, cavity, gap, breach, tunnel","mwahuha [mwahuuha], mi-, n. (vinr), a yawning ..."
61,61,P5-C1,-ahubi [-ahuuhi],None,None,a.,restless,"-ahubi [-ahuuhi], a. (1), restless; mũndũ mw~,..."
62,62,P5-C1,mwahuhĩko [mwahuuhiko],mi-,None,n.,habit of nosing about,"mwahuhĩko [mwahuuhiko], mi-, n. (1), habit of ..."
63,63,P5-C1,mwahunga,mi-,None,n.,"a yawning hole, tunnel, wide opening, or crack","mwahunga, mi-, n. (1v), a yawning hole, tunnel..."
64,64,P5-C1,mwahũ,mi-,None,n.,hand of bananas,"mwahũ, mi-, n. (v), hand of bananas; cf. gĩkah..."


In [417]:
kikuyu_tagged_df.loc[kikuyu_tagged_df["INDEX"] == 4291]

,INDEX,PAGE-COLUMN,WORD,PREFIX,NC,POS,DESCRIPTION,FULL_ENTRY
4291,4291,P250-C1,kwini,ma-,1/6 or 1/2,n.,queen.,"kwini, ma-, n. cl. 1/6 or 1/2 (1x), queen. [E...."


In [85]:
kikuyu_tagged_df.to_csv("kikuyu_tagged_full.csv", index=False)

In [86]:
# search_str = r"\),\s"
search_str = r"cl\.\s*(\d[\w\s]*?)(?:,|\(|\.)"

nc_kikuyu_df = kikuyu_tagged_df[kikuyu_tagged_df["FULL_ENTRY"].str.contains(search_str)]

# nc_kikuyu_df['FULL_ENTRY'].tolist()

/var/folders/lc/yr0c0sq938g9gtrdrvgrk6nm0000gn/T/ipykernel_91345/1294994596.py:5: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  kikuyu_tagged_df['FULL_ENTRY'].str.contains(search_str)


In [67]:
# nc_kikuyu_df.to_csv('kikuyu_nc_check.csv',index=False)

## 3.1 Cleanup

In [418]:
def fix_word_column(df):
    def fix_word(word):
        # Rule 1: trailing "!" → append (1), remove "!"
        # Rule 2: trailing "?" → append (2), remove "?"
        # Rule 3: trailing digit → append (digit in parens), remove digit
        # All rules preserve anything after the word (e.g. bracketed alternates)

        # Split word from any trailing bracketed content e.g. " [mũkiyo]"
        bracket_match = re.search(r"(\s*\[.*\].*)$", word)
        tail = bracket_match.group(1) if bracket_match else ""
        core = word[: bracket_match.start()] if bracket_match else word

        if ("!" in core) or ("'" in core):
            core = core[:-1].rstrip() + " (1)"
        elif "?" in core:
            core = core[:-1].rstrip() + " (2)"
        elif re.search(r"\d$", core):
            digit = re.search(r"(\d+)$", core).group(1)
            core = re.sub(r"\d+$", "", core).rstrip() + f" ({digit})"

        return core + tail

    df["WORD"] = df["WORD"].apply(fix_word)
    return df


kikuyu_tagged_df = fix_word_column(kikuyu_tagged_df)

In [88]:
kikuyu_tagged_df.to_csv("kikuyu_tagged_full.csv", index=False)

### Clean NC tags

In [419]:
# strategy: make a gold standard label set of all possible NC annotations??
# 1. Filter valid rows
subset = kikuyu_tagged_df[
    kikuyu_tagged_df["NC"].notna() & (kikuyu_tagged_df["NC"] != "")
]

# 2. Calculate frequencies and sort by most frequent
counts = subset["NC"].value_counts()

# 3. Get one sample row for each noun class
# We take the first appearance and join the frequency count
result = subset.drop_duplicates(subset="NC", keep="first").copy()
result["FREQUENCY"] = result["NC"].map(counts)

# 4. Sort by frequency (descending) and select columns
result = result.sort_values(by="FREQUENCY", ascending=False)
final = result[["INDEX", "WORD", "NC", "PAGE-COLUMN", "FREQUENCY"]]

# 5. Export
final.to_csv("kikuyu_NC_freq_subset.csv", index=False)

In [420]:
# lookup any NC
subset[subset["NC"] == "r14"][["NC", "PAGE-COLUMN", "FULL_ENTRY"]].head(30)

,NC,PAGE-COLUMN,FULL_ENTRY
4996,r14,P280-C2,"ũni, n. cl. r14 only (1), pith, spongy centre ..."
9791,r14,P538-C1,"thũti [thũti], ~, n. (1?) mean, stingy, person..."


In [426]:
# CLEAN THE ENTIRE DICT using mapping !

# 1. Load the mapping file
mapping_df = pd.read_csv("kikuyu_NC_freq_subset_mapped.csv")

# 2. Filter out rows where NOUN_CLASS_ACTUAL is empty/NaN
valid_map = mapping_df.dropna(subset=["NC_FIXED"])
valid_map = valid_map[valid_map["NC_FIXED"].str.strip() != ""]

# 3. Create the dictionary from valid pairs only
mapping_dict = dict(zip(valid_map["NC"], valid_map["NC_FIXED"]))

# 4. Apply mapping: if key not in dict, .map() returns NaN, then .fillna() restores original
kikuyu_tagged_df["NC"] = (
    kikuyu_tagged_df["NC"].map(mapping_dict).fillna(kikuyu_tagged_df["NC"])
)

In [427]:
# get a list of ALL POSSIBLE NC TAGS, without separating into individual - just to verify the cleanup works.

# 1. Force everything to string and strip whitespace to ensure '14' == 14
effective_labels = (
    mapping_df["NC_FIXED"].fillna(mapping_df["NC"]).astype(str).str.strip()
)

# 2. Filter out common "empty" string representations
effective_labels = effective_labels[~effective_labels.isin(["nan", "None", ""])]

# leave out all OCR labels with freq == 1
nc_set_gold = set(effective_labels.tolist()[:])

nc_set_gold

{'1',
 '1 and 7',
 '1/2',
 '1/2 or 6',
 '1/2 or 7',
 '1/2 or 9/10',
 '1/6',
 '1/6 or 1/2',
 '10',
 '11',
 '11 or 14',
 '11 or 14/6',
 '11 or 9',
 '11/10',
 '11/10 or 6',
 '11/6',
 '12',
 '12/13',
 '13',
 '14',
 '14 or 5/6',
 '14 or 9',
 '14 or 9/6',
 '14/10',
 '14/10 or 6',
 '14/2 or 6',
 '14/6',
 '14/6 or 1/2',
 '14/6 or 10',
 '14/6 or 9/10',
 '15',
 '15/6',
 '15b',
 '16',
 '17',
 '2',
 '3',
 '3/4 or 6',
 '4',
 '4 and cl',
 '5',
 '5 or cl',
 '5/6',
 '5/6 or 1/2',
 '6',
 '6 or 10',
 '6 or 4',
 '6 or 8',
 '7',
 '7 or 14',
 '7 or 15b',
 '8',
 '8 and 10',
 '9',
 '9 or 1',
 '9/10',
 '9/10 or 1/2',
 '9/10 or 14/6',
 '9/10 or 6',
 '9/6',
 '9/6 or 1/2'}

In [428]:
# final save

kikuyu_tagged_df.to_csv("kikuyu_tagged_full.csv", index=False)

### Fix empty DESC

In [429]:
kikuyu_tagged_df = pd.read_csv("kikuyu_tagged_full.csv")

In [430]:
# fix empty description
if subset_df is not None: del subset_df
    
subset_df = kikuyu_tagged_df[
    kikuyu_tagged_df["DESCRIPTION"].isna() | (kikuyu_tagged_df["DESCRIPTION"] == "1.")
].copy()

In [400]:
# if full_entry has no pos tag, and has only one full stop, or only two commas, then mark this as "reference" and move on.
# for the others (filter by not "reference"), extract a description.

In [431]:
ref_list_df = subset_df[
    (
        (subset_df["POS"].isna())
        & (subset_df["FULL_ENTRY"].notna())
        & ((subset_df["FULL_ENTRY"].str.count("\\.") == 1))
        | (subset_df["FULL_ENTRY"].str.contains(", see"))
    )
]

# ref_list_df["FULL_ENTRY"].tolist()

In [402]:
len(ref_list)

1222

In [432]:
len(
    [e for e in ref_list if ", see" in e]
)  # there were only 17 real entries that we miss among the 200, so letting this condition be the OR check.

1195

In [433]:
ref_list_df.loc[:, "DESCRIPTION"] = "REFERENCE"

In [434]:
ref_list_df

,INDEX,PAGE-COLUMN,WORD,PREFIX,NC,POS,DESCRIPTION,FULL_ENTRY
7,7,P1-C1,aari,NaN,NaN,NaN,REFERENCE,"aari, see naari."
10,10,P1-C2,Abaci,NaN,NaN,NaN,REFERENCE,"Abaci, see Habaci."
15,15,P1-C2,aboita,NaN,NaN,NaN,REFERENCE,"aboita, see abaita."
18,18,P2-C1,acũhia,NaN,NaN,NaN,REFERENCE,"acũhia, see icũhia,"
88,88,P6-C2,akorũo,NaN,NaN,NaN,REFERENCE,"akorũo, see under kora."
...,...,...,...,...,...,...,...,...
10136,10136,P560-C2,mawariũngũ,NaN,NaN,NaN,REFERENCE,"mawariũngũ, see under rũariũngũ."
10141,10141,P561-C1,warũbũkũ,NaN,NaN,NaN,REFERENCE,"warũbũkũ, see under mbũkg."
10144,10144,P561-C1,wayũa (N.K.),NaN,NaN,NaN,REFERENCE,"wayũa (N.K.), see aiwa.,"
10157,10157,P562-C1,yaya (2),NaN,NaN,NaN,REFERENCE,"yaya?, see waya."


In [435]:
subset_df.update(
    ref_list_df, overwrite=True
)  # overwrites all new entries excluding NaNs

In [436]:
entries = subset_df[subset_df["DESCRIPTION"] != "REFERENCE"]["FULL_ENTRY"].tolist()

In [437]:
def is_noun_entry(entry):
    return " n." in entry


GIKUYU_CHARS = re.compile(r"[ĩũʻ]")

abbr_list = pos_tags + ['expr.', 'ref.','N.K.','I. ','descr.','arch.','r.','x.','indic.','xĩ.','cf.','Bibl.','sp.','dim.','sth.']

def is_gikuyu_lead_clause(clause):
    """True if this leading clause is a Gikuyu example phrase/word, not English gloss text."""
    if not clause:
        return True
    if clause.startswith("~"):
        return True
    if GIKUYU_CHARS.search(clause):
        return True
    if re.match(r"^-?[a-z']{1,4}\s*~$", clause):
        return True
    if re.match(
        r"^\(used\s", clause, re.IGNORECASE
    ):  # "(used adv.)" style parenthetical lead
        return True
    if re.match(r"^(see|under)\b", clause, re.IGNORECASE):
        return True
    return False


def find_class_zone_end(entry):
    """Return index right after the noun-class apparatus ends. This extraction is focused on nouns.
    The content INSIDE (...) is never meaningful for the description (it's a class
    code, possibly OCR-garbled like (xX), (V11), (vin), (xI-IH1)} -- so we just need
    to find where that bracketed zone closes, not validate what's inside it."""
    # explicit "cl. ... ;" form
    m = re.search(r"cl\.[^;]*?(?:\)|,)\s*;?", entry)
    if m:
        end = m.end()
        if end < len(entry) and entry[end] == ";":
            end += 1
        return end
    # ANY parenthetical right after "n." -- don't validate contents, just find the close.
    # Search starting from the "n." marker so we don't grab an earlier unrelated "(...)"
    # e.g. "gĩcinicini [gĩcinicini], i-, n. (xX);" -- skip the leading "[gĩcinicini]" bracket.
    n_marker = re.search(r"\bn\.\s*", entry)
    search_from = n_marker.end() if n_marker else 0
    m = re.search(r"\(([^()]*)\)[\}\)]?", entry[search_from:])
    if m:
        return search_from + m.end()
    # bare class code with no parens at all, e.g. "n. TII, 1. ..."
    # handled by the "1." anchor below, so just fall back to after "n."
    if n_marker:
        return n_marker.end()
    return 0


import re

def remove_abbreviations(entry, abbr_list):
    sorted_abbr = sorted(abbr_list, key=len, reverse=True)
    pattern = "|".join(re.escape(a) for a in sorted_abbr)
    cleaned = re.sub(rf"\b(?:{pattern})\.*", "", entry)
    cleaned = re.sub(r"\s{2,}", " ", cleaned).strip()
    return cleaned

def extract_first_sense(entry):
    start = find_class_zone_end(entry)
    rest = entry[start:].strip(" ,;")

    # Keep a raw copy with abbreviations intact for final description cutting
    original_rest = rest  

    rest = remove_abbreviations(' ' + rest, abbr_list)

    m1 = re.search(r"1\.\s*", rest)
    if m1:
        rest = rest[m1.end() :]
    else:
        rest = re.sub(r"^1\.\s*", "", rest)

    while True:
        m = re.match(r"^([^,;.]+)[,;]\s*", rest)
        if m and is_gikuyu_lead_clause(m.group(1).strip()):
            rest = rest[m.end() :]
            continue
        break

    rest = re.sub(
        r"^\((?:used|usu\.?|esp\.?|chiefly|mostly)\b[^()]*\)\s*",
        "",
        rest,
        flags=re.IGNORECASE,
    )

    # --- FIXED TERMINATION LOGIC ---
    # Scan characters one by one to find the true sentence terminator
    search_idx = 0
    while search_idx < len(rest):
        # Find the next potential terminator
        m_term = re.search(r"[.;]", rest[search_idx:])
        if not m_term:
            # No more terminators found, take everything left
            search_idx = len(rest)
            break
            
        term_pos = search_idx + m_term.start()
        
        if rest[term_pos] == ';':
            # Semicolons are never part of abbreviations; stop here safely
            search_idx = term_pos
            break
            
        if rest[term_pos] == '.':
            # Check if this period is part of an abbreviation
            matched_abbr = None
            # Look backwards from the period to see if an abbreviation matches
            for abbr in sorted(abbr_list, key=len, reverse=True):
                # If the abbreviation ends with a period, locate where it would start
                if abbr.endswith('.'):
                    start_look = term_pos - len(abbr) + 1
                    if start_look >= 0 and rest[start_look:term_pos + 1].lower() == abbr.lower():
                        matched_abbr = abbr
                        break
            
            if matched_abbr:
                # Skip past the remaining expected periods of this abbreviation
                # e.g., if N.K. is found, skip past its length so we don't trip on its internal periods
                search_idx = start_look + len(matched_abbr)
            else:
                # It's a real terminal full stop!
                search_idx = term_pos
                break

    desc = rest[:search_idx]
    return desc.strip(" ,;.")

# print(f"{len(entries)} entries\n")
# for e in entries:
#     print(repr(e[:100]), "->\n", repr(extract_first_sense(e)), "\n\n")

In [438]:
# 1. Compute the extracted senses across the entire subset_df
extracted_series = subset_df[
    (subset_df["DESCRIPTION"] != "REFERENCE") & 
    (subset_df["FULL_ENTRY"].notna())
]["FULL_ENTRY"].apply(extract_first_sense)

# 2. TARGET LOGIC: Select every index in subset_df EXCEPT the ones in ref_list_df
ref_indices = ref_list_df["INDEX"].tolist()
target_indices = subset_df.index.difference(ref_indices)

# 3. Intersect targets with your extracted rows to avoid any mapping gaps
valid_targets = extracted_series.index.intersection(target_indices)

# 4. Safely update DESCRIPTION directly using .loc
subset_df.loc[valid_targets, "DESCRIPTION"] = extracted_series.loc[valid_targets]

In [439]:
subset_df.to_csv("subset_df.csv", index=False)

In [440]:
# apply new DESC logic to the ENTIRE dataset

# 1. Compute the extracted senses across the entire subset_df
extracted_series = kikuyu_tagged_df[
    (kikuyu_tagged_df["FULL_ENTRY"].notna())
]["FULL_ENTRY"].apply(extract_first_sense)

# 2. TARGET LOGIC: Select every index in subset_df EXCEPT the ones in subset_df
subset_indices = subset_df["INDEX"].tolist()
target_indices = kikuyu_tagged_df.index.difference(subset_indices)

# 3. Intersect targets with your extracted rows to avoid any mapping gaps
valid_targets = extracted_series.index.intersection(target_indices)

# 4. Safely update DESCRIPTION directly using .loc
kikuyu_tagged_df.loc[valid_targets, "DESCRIPTION"] = extracted_series.loc[valid_targets]

In [441]:
# finally, merge subset_df into the main df

# 1. Identify the rows in the main DataFrame that exist in the subset
overlapping_indices = kikuyu_tagged_df.index.intersection(subset_df.index)

# 2. Overwrite those exact rows entirely with subset_df's data
kikuyu_tagged_df.loc[overlapping_indices] = subset_df.loc[overlapping_indices]

In [442]:
kikuyu_tagged_df.to_csv("kikuyu_complete_cleaned.csv", index=False)

In [443]:
import pandas as pd

def generate_description_diffs(file1_path, file2_path):
    # 1. Load the CSV files
    # Note: If your CSVs have a specific ID/Index column, add: index_col="YOUR_INDEX_COL"
    df1 = pd.read_csv(file1_path)
    df2 = pd.read_csv(file2_path)
    
    # 2. Ensure they are aligned on the same shape and indices
    if len(df1) != len(df2):
        print(f"Warning: File row counts differ ({len(df1)} vs {len(df2)}). Diffs will align on row index.")

    # 3. Create a boolean mask where the descriptions do not match
    # (Handling potential NaN/Null values safely using .ne)
    changed_mask = df1['DESCRIPTION'].ne(df2['DESCRIPTION'])
    
    # 4. Filter out the rows that changed
    diff_df = pd.DataFrame({
        'ROW_INDEX': df1[changed_mask].index,
        'ORIGINAL_VALUE': df1.loc[changed_mask, 'DESCRIPTION'],
        'NEW_VALUE': df2.loc[changed_mask, 'DESCRIPTION'],
        'FULL_ENTRY': df1.loc[changed_mask, 'FULL_ENTRY']
    })
    
    # 5. Output results
    total_changes = len(diff_df)
    print(f"=== Comparison Report ===")
    print(f"Total number of row changes: {total_changes}\n")
    
    # Optional: Save the diff report to a new CSV file
    diff_df.to_csv('desc_update_diff.csv', index=False)
    
    return diff_df

# --- Run the function ---
generate_description_diffs('kikuyu_tagged_full.csv', 'kikuyu_complete_cleaned.csv')

=== Comparison Report ===
Total number of row changes: 7061



,ROW_INDEX,ORIGINAL_VALUE,NEW_VALUE,FULL_ENTRY
0,0,"n., a., & p.c.",a,"a- ] (N.K.) ma-, conc. prefix (U), n., a., & p..."
1,1,"ref. to cL 2, who (exactly) ?, which people?","to cL 2, who (exactly) ?, which people?","a / ma, interrog. pron. (U), ref. to cL 2, who..."
2,2,"indic. relationship, association, or possessio...","relationship, association, or possession, prec...","-a, part. (U), indic. relationship, associatio..."
3,3,sign of the immediate past tenses.,sign of the immediate past tenses,"-a-, v. infix (U), sign of the immediate past ..."
4,4,"expr. deep disgust, irritation","deep disgust, irritation","a’ | aa [aa-a], interj. (U), expr. deep disgus..."
...,...,...,...,...
10161,10161,"moss, lichen.","moss, lichen","kĩyo?, i-, n. (1), moss, lichen."
10162,10162,"succulent plant, growing on abandoned village ...","succulent plant, growing on abandoned village ...","mũyuyu, mĩ-, n. (v11), succulent plant, growin..."
10163,10163,1. strong voice,"power (primarily of voice), vitality, gaiety","yũ / iyũ, ma-, n. (111), 1. strong voice; ĩta ..."
10164,10164,"stem of dem. cl. 1, 3, and 14, this (here, clo...","3, and 14, this (here, close at hand, near the...","-yũ, p.a. (U), stem of dem. cl. 1, 3, and 14, ..."
